# Commercial vertical and brands (LLM)

After [intent_classification_llm.ipynb](intent_classification_llm.ipynb) has produced labelled categories/subcategories, this notebook filters to **commercial_investigation** only, then in a single LLM pass adds:

- **Commercial vertical** labels (`vertical_tier1_llm`, `vertical_tier2_llm`) — LLM-only, no intent re-classification.
- **Brands** mentioned in the conversation, with `where`: `query_only`, `answer_only`, or `both`.

Uses **all queries and all answers** per conversation. Output is saved to parquet.

In [ ]:
# Control variables (edit and run first)
INTENT_OUTPUT_DIR = "intent_output"
VERTICAL_OUTPUT_DIR = "vertical_output"   # output folder (not inside intent_output)
CATEGORY_FILTER = "informational"
PROCESS_ALL = True   # if False, use index range below
START_INDEX = 0
END_INDEX = 20000   # used when PROCESS_ALL is False; output folder name will be 0_20000

# Subcategory: load by intent_sub (e.g. education) instead of major category
USE_SUBCATEGORY = True
SUBCATEGORY_NAME = "education"   # used when USE_SUBCATEGORY is True (by_sub/<name>.parquet)

# Sampling: take a random sample of N rows (ignores index range when used)
USE_SAMPLE = True
SAMPLE_N = 5000
RANDOM_SEED = 42

LLM_BATCH_SIZE = 50
LLM_MAX_WORKERS = 50
USE_CACHE = True
SHOW_PROGRESS = True

# Derived: range label for output folder and cache (e.g. 0_20000 or "all")
RANGE_LABEL = "all" if PROCESS_ALL else f"{START_INDEX}_{END_INDEX}"
CACHE_PATH = f"{VERTICAL_OUTPUT_DIR}/cache_commercial_vertical_brands_{RANGE_LABEL}.parquet"

## Load commercial data

Load by major category (e.g. commercial_investigation) or by **subcategory** (e.g. education from by_sub). Use PROCESS_ALL or index range; optionally set USE_SAMPLE and SAMPLE_N to take a random sample of N rows. Add **all_queries** and **all_answers**.

In [2]:
from pathlib import Path

from eda_utils import ensure_conversation_parsed
from intent_analysis_utils import ensure_conversation_normalized
from intent_taxonomy import load_commercial_from_intent_output
from commercial_vertical_utils import add_all_queries_answers_columns

category_name = SUBCATEGORY_NAME if USE_SUBCATEGORY else CATEGORY_FILTER
which = "sub" if USE_SUBCATEGORY else "major"

if PROCESS_ALL:
    df = load_commercial_from_intent_output(
        INTENT_OUTPUT_DIR, category=category_name, which=which
    )
else:
    df = load_commercial_from_intent_output(
        INTENT_OUTPUT_DIR,
        category=category_name,
        start_index=START_INDEX,
        end_index=END_INDEX,
        which=which,
    )

if USE_SAMPLE and SAMPLE_N and len(df) > SAMPLE_N:
    df = df.sample(n=SAMPLE_N, random_state=RANDOM_SEED).reset_index(drop=True)
    print(f"Sampled {SAMPLE_N} rows (random_state={RANDOM_SEED}).")

df = ensure_conversation_parsed(df)
df = ensure_conversation_normalized(df)
df = add_all_queries_answers_columns(df, conversation_col="conversation")
print(f"Loaded {len(df)} {category_name} rows. Columns: {list(df.columns)}")
if len(df) > 0:
    df[["conversation_id", "all_queries", "all_answers"]].head(2)

Sampled 5000 rows (random_state=42).
Loaded 5000 education rows. Columns: ['conversation_id', 'model', 'timestamp', 'conversation', 'turn', 'language', 'openai_moderation', 'detoxify_moderation', 'toxic', 'redacted', 'text', 'intent_major', 'intent_sub', 'all_queries', 'all_answers']


## Run LLM (vertical + brands)

Call OpenAI with all_queries + all_answers per conversation; get structured JSON with `vertical_tier1`, `vertical_tier2`, and `brands` (name + where). Parallel, cached, with progress bar.

In [3]:
if len(df) == 0:
    print("No rows to label. Skip LLM and save cell or run with data.")
else:
    from dotenv import load_dotenv
    load_dotenv()

    from commercial_vertical_utils import label_vertical_brands_llm_parallel

    df = label_vertical_brands_llm_parallel(
        df,
        queries_col="all_queries",
        answers_col="all_answers",
        cache_path=CACHE_PATH,
        use_cache=USE_CACHE,
        batch_size=LLM_BATCH_SIZE,
        max_workers=LLM_MAX_WORKERS,
        show_progress=SHOW_PROGRESS,
    )
    print("Vertical (LLM) Tier 1 value counts:")
    display(df["vertical_tier1_llm"].value_counts().head(15))

LLM vertical+brands: 100%|██████████| 100/100 [28:27<00:00, 17.07s/batch]  

Vertical (LLM) Tier 1 value counts:


vertical_tier1_llm
Other           1437
Education       1355
Technology      1012
Health           600
Finance          253
Food & Drink      85
Sports            75
Automotive        50
Travel            50
Real Estate       34
Shopping          21
Music              9
Politics           3
Aviation           3
Agriculture        3
Name: count, dtype: int64

## Brand mention dynamics

After LLM vertical+brands, compute per-brand metrics (first_mention_round, first_mention_role, mention_count_total, last_mention_round, mention_span, user_follow_up, brand_mentioned_once, conversation_ended_soon_after) and per-conversation summary columns. Adds `brands_enriched` and summary columns; keeps `brands`.

In [4]:
if len(df) > 0 and "brands" in df.columns and "conversation" in df.columns:
    from commercial_vertical_utils import add_brand_mention_dynamics
    df = add_brand_mention_dynamics(
        df,
        conversation_col="conversation",
        brands_col="brands",
        ended_soon_threshold=2,
    )
    print("Added brands_enriched and summary columns (first_brand_first_round, any_brand_user_introduced, etc.).")
else:
    print("Skip: need conversation and brands columns (run Load and LLM first).")

Added brands_enriched and summary columns (first_brand_first_round, any_brand_user_introduced, etc.).


## Inspect brands

Sample rows with non-empty brands; optional derived columns for query_only / answer_only / both.

In [5]:
if len(df) > 0 and "brands" in df.columns:
    with_brands = df[df["brands"].apply(lambda x: (len(x) if isinstance(x, list) else 0) > 0)]
    print(f"Rows with at least one brand: {len(with_brands)}")
    if len(with_brands) > 0:
        display(with_brands[["conversation_id", "vertical_tier1_llm", "brands"]].head(5))
else:
    print("No brands column or empty df.")

Rows with at least one brand: 1136


,conversation_id,vertical_tier1_llm,brands
12,0b8aa6a89c785d565dfcc146648a0673,Technology,"[{'name': 'Oracle', 'where': 'answer_only'}]"
15,e09b00a1b7f5b7084f8bac4b4e534c20,Technology,"[{'name': 'PyTorch', 'where': 'answer_only'}]"
21,ced77ce5e543ee919d51a7f89810099c,Sports,"[{'name': 'Madden NFL', 'where': 'both'}, {'na..."
23,af70ed319992bdf3b63db2e3d4a17a1b,Technology,"[{'name': 'Wien2k', 'where': 'both'}]"
32,e6dd5ade4196256e78947198921f0be6,Education,"[{'name': 'Coursera', 'where': 'both'}, {'name..."


In [6]:
# Check if a specific conversation_id is in the df with brands
CHECK_CONVERSATION_ID = "f7d908abbfabc8acebd13ffda10e3e29"  # Replace with the conversation_id you want to check

if len(df) > 0 and "brands" in df.columns:
    match = df[df["conversation_id"] == CHECK_CONVERSATION_ID]
    if len(match) == 0:
        print(f"conversation_id {CHECK_CONVERSATION_ID!r} not found in table.")
    else:
        brands = match.iloc[0]["brands"]
        print(f"conversation_id {CHECK_CONVERSATION_ID!r} found in table.")
        if brands and isinstance(brands, list) and len(brands) > 0:
            print("Brands for this conversation:")
            for b in brands:
                print(b)
        else:
            print("No brands for this conversation.")
else:
    print("No brands column or empty df.")

conversation_id 'f7d908abbfabc8acebd13ffda10e3e29' not found in table.


## Spot check by vertical

Select a vertical and sample a few conversations to inspect. Set `SPOTCHECK_VERTICAL` (must match a value in `vertical_tier1_llm`) and `SPOTCHECK_N` (number to sample), then run the cell below.

In [7]:
# Spot check: select vertical and sample size
SPOTCHECK_VERTICAL = "Technology"   # must match a value in df["vertical_tier1_llm"]
SPOTCHECK_N = 3

if len(df) == 0 or "vertical_tier1_llm" not in df.columns:
    print("No data or no vertical_tier1_llm; run Load and LLM cells first.")
else:
    from intent_analysis_utils import format_conversation
    import random

    available = df["vertical_tier1_llm"].dropna().unique().tolist()
    print("Available verticals:", available)
    sub = df[df["vertical_tier1_llm"] == SPOTCHECK_VERTICAL]
    if len(sub) == 0:
        print(f"No rows for vertical {SPOTCHECK_VERTICAL!r}. Pick one from the list above.")
    else:
        n = min(SPOTCHECK_N, len(sub))
        indices = random.Random(42).sample(list(sub.index), n)
        for idx in indices:
            row = df.loc[idx]
            cid = row.get("conversation_id", idx)
            print("=" * 60)
            print(f"conversation_id: {cid}  |  vertical: {row.get('vertical_tier1_llm')}")
            print("=" * 60)
            print(format_conversation(row.get("conversation"), width=80))
            print()

Available verticals: ['Health', 'Sports', 'Technology', 'Education', 'Other', 'Legal', 'Finance', 'Food & Drink', 'Shopping', 'Politics', 'Automotive', 'Music', 'Aviation', 'Real Estate', 'Travel', 'History', 'Construction', 'Transportation', 'Agriculture', 'Home & Garden', 'Marketing']
conversation_id: c4f473dabf6c61bc7a80af2f87c46a09  |  vertical: Technology
[user]
Question No: 93 What   is   the   privilege   required   to   connect/login
from   SQL   Developer   for   a   new   user   ?  Options:  ALL   Privileages
CREATE   OBJECT  CREATE   SESSION  GRANT   Privilege

[assistant]
The privilege required to connect/login from SQL Developer for a new user is
"CREATE SESSION".

[user]
Question No: 94 Choose   the   correct   syntax   for   creating   table:
Options:  CREATE   TABLE   tablename()   Values();  CREATE   TABLE
tablename(Col1   Datatype,   col2   Datatype,...,coln   Datatype);  CREATE
TABLE   tablename   Values();  CREATE   TABLE   tablename()
Values(value1,value2,val3);

[

## Save to parquet

Write full table (all original columns + vertical_tier1_llm, vertical_tier2_llm, brands, brands_enriched, summary columns, product_mentioned, product_inferred, deal_size_usd) to parquet. Brands and brands_enriched stored as JSON strings for compatibility.

In [8]:
from pathlib import Path
import json

from insights_utils import ensure_output_dir

if len(df) == 0:
    print("No rows; skipping save.")
else:
    category_label = CATEGORY_FILTER.replace(" ", "_")
    out_dir = Path(VERTICAL_OUTPUT_DIR) / RANGE_LABEL
    out_path = out_dir / f"commercial_vertical_brands_{category_label}_{RANGE_LABEL}.parquet"
    ensure_output_dir(out_dir)
    out_df = df.copy()
    if "brands" in out_df.columns:
        out_df["brands"] = out_df["brands"].apply(
            lambda x: json.dumps(x) if isinstance(x, list) else (x if isinstance(x, str) else "[]")
        )
    if "brands_enriched" in out_df.columns:
        out_df["brands_enriched"] = out_df["brands_enriched"].apply(
            lambda x: json.dumps(x) if isinstance(x, list) else (x if isinstance(x, str) else "[]")
        )
    out_df.to_parquet(out_path, index=False)
    print(f"Saved {len(out_df)} rows to {out_path}.")

Saved 5000 rows to vertical_output/all/commercial_vertical_brands_informational_all.parquet.
